In [ ]:
import featureform as ff

client = ff.Client(host="localhost:7878", insecure=True)

## Регистрируем Провайдеры

In [ ]:
postgres = ff.register_postgres(
    name="postgres-quickstart",
    host="0.0.0.0",
    port="5432",
    user="postgres",
    password="password",
    database="transactions",
)
# cassandra = ff.register_cassandra(
#     name="cassandra-provider",
#     host="127.0.0.1",
#     port=9042,
#     username="",
#     password="",
#     keyspace="featureform",
#     replication=1,
#     consistency="ANY",
# )


redis = ff.register_redis(
    name="redis-quickstart",
    host="0.0.0.0",
    port=6379,
)
client.apply()

## Заполнение данных

In [ ]:
from datetime import datetime, timezone

variant = datetime.now(tz=timezone.utc).strftime("%s")

print(variant)


In [ ]:

transactions = postgres.register_table(
    name="transactions", table="transactions", variant=variant
)


@postgres.sql_transformation(inputs=[transactions], variant=variant)
def average_user_transaction(tr):
    return (
        "SELECT CustomerID as user_id, avg(TransactionAmount) "
        "as avg_transaction_amt from {{tr}} GROUP BY user_id"
    )


@ff.entity
class User:
    avg_transactions_feature = ff.Feature(
        average_user_transaction[["user_id", "avg_transaction_amt"]],
        type=ff.Float32,
        inference_store=redis,
        variant=variant,
    )

    is_suspicious = ff.Label(
        transactions[["customerid", "isfraud", "timestamp"]],
        type=ff.Bool,
        variant=variant,
    )


client.apply()

## Training set

In [ ]:
tr_set = ff.register_training_set(
    name="fraud_training",
    label=User.is_suspicious,
    features=[User.avg_transactions_feature],
    variant=variant,
)

client.apply()

In [ ]:
dataset = client.training_set(tr_set.name, tr_set.variant)

df = dataset.dataframe()

In [ ]:
df.sample(4)

## Getting Features

In [ ]:
user = client.get_entity("user")

In [ ]:
user

In [ ]:
client.dataframe(transactions)

In [ ]:
USER = {"user": "C5342380"}
user_feat = client.features(
    [User.avg_transactions_feature.name_variant()], entities=USER
)
print(f"User Result: {user_feat}")

## Creating On-Demand Transformation

In [ ]:
User.avg_transactions_feature.name_variant()

In [ ]:
@ff.ondemand_feature()
def add_val(client, params, entities):
    user_avg = client.features(
        [("avg_transactions_feature", "1717247205")], entities=entities
    )
    return user_avg[0] + params[0]


client.apply()

In [ ]:
feats = client.features(
    features=[
        User.avg_transactions_feature.name_variant(),
        add_val,
    ],
    entities=USER,
    params=[1000],
)
print(f"Before transformation: {feats[0]}; After: {feats[1]}")